# 2 · Silver — transformación con PySpark + Iceberg (`silver_transform.ipynb`)

Respaldo manual de la tarea **`silver_transform`** del DAG. Ejecuta **la misma función** que Airflow (`jobs/silver_transform.py`).

- Lee `votes_2020` (subido a mano) y `votes_2021` (traído por dlt) de Bronze.
- Cura y normaliza (snake_case, tipos, catálogo de tipos de voto, `UserId = 0` → `NULL`, sin duplicados) y agrega **`fecha_cargue`**.
- Consolida en la tabla histórica **`nessie.silver.votes_hist`** (Iceberg) con **`MERGE INTO`** (no overwrite).

La tabla histórica de posts (`nessie.silver.post_hist`) se construye con un **script manual** (`jobs/silver_post_hist_manual.py`), como permite el enunciado. Si todavía no existe, se puede crear con la celda opcional del final.

In [1]:
NOMBRE_APP = "silver_transform_notebook"
import os, sys
sys.path.insert(0, os.environ.get("JOBS_DIR", "/workspace/jobs"))   # carpeta jobs/ del proyecto
import comun

# Sesión de Spark contra el clúster (spark://spark-master:7077): aparece en la UI de Spark (9090)
spark = comun.crear_spark(NOMBRE_APP, master=os.environ["SPARK_MASTER"])
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/14 15:16:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Transformación y MERGE en `nessie.silver.votes_hist`

In [2]:
import silver_transform
silver_transform.main(spark)

26/10/14 15:16:47 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Bronze s3a://bronze/votes/votes_2020/: 4,839,670 filas


Bronze s3a://bronze/votes/votes_2021/: 4,831,211 filas


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


nessie.silver.votes_hist: 9,670,881 filas


+-----------------------+-------------------+---------+---------+--------+
|committed_at           |snapshot_id        |operation|agregadas|borradas|
+-----------------------+-------------------+---------+---------+--------+
|2026-10-14 15:18:12.912|8537173846617496089|overwrite|9670881  |9670881 |
|2026-10-14 15:02:59.09 |2024969404979295706|overwrite|9670881  |NULL    |
+-----------------------+-------------------+---------+---------+--------+



+----+-------+
|anio|  votos|
+----+-------+
|2020|4839670|
|2021|4831211|
+----+-------+



9670881

## Verificación

In [3]:
spark.sql(f"""
    SELECT vote_type, COUNT(*) AS votos
    FROM {comun.TABLA_VOTES_HIST}
    GROUP BY vote_type ORDER BY votos DESC
""").show()
spark.table(comun.TABLA_VOTES_HIST).orderBy("id").show(5, truncate=False)

+------------------+-------+
|         vote_type|  votos|
+------------------+-------+
|     voto_positivo|7621520|
|     voto_negativo| 973878|
|           borrado| 425699|
|respuesta_aceptada| 404667|
|  edicion_aprobada| 156921|
|revision_moderador|  39128|
|      restauracion|  20878|
| inicio_recompensa|  12077|
| cierre_recompensa|  11792|
|              spam|   3550|
|          ofensivo|    771|
+------------------+-------+



+---------+--------+------------+-------------+-------------------+-------+-------------+----+---+-------------------------+
|id       |post_id |vote_type_id|vote_type    |creation_date      |user_id|bounty_amount|anio|mes|fecha_cargue             |
+---------+--------+------------+-------------+-------------------+-------+-------------+----+---+-------------------------+
|200597588|59542132|3           |voto_negativo|2020-01-01 00:00:00|NULL   |0            |2020|1  |2026-10-14 15:02:20.41567|
|200597589|665037  |2           |voto_positivo|2020-01-01 00:00:00|NULL   |0            |2020|1  |2026-10-14 15:02:20.41567|
|200597592|3786774 |2           |voto_positivo|2020-01-01 00:00:00|NULL   |0            |2020|1  |2026-10-14 15:02:20.41567|
|200597593|13580577|2           |voto_positivo|2020-01-01 00:00:00|NULL   |0            |2020|1  |2026-10-14 15:02:20.41567|
|200597594|3786790 |2           |voto_positivo|2020-01-01 00:00:00|NULL   |0            |2020|1  |2026-10-14 15:02:20.41567|


In [4]:
# Historial de la tabla: cada MERGE deja un snapshot nuevo en Iceberg
spark.sql(f"SELECT committed_at, snapshot_id, operation FROM {comun.TABLA_VOTES_HIST}.snapshots ORDER BY committed_at").show(truncate=False)

+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-10-14 15:02:59.09 |2024969404979295706|overwrite|
|2026-10-14 15:18:12.912|8537173846617496089|overwrite|
+-----------------------+-------------------+---------+



## (Opcional) Tabla histórica de posts — script manual

Solo hace falta la primera vez, después de subir `posts_2020` y `posts_2021` a Bronze (`scripts/bronze_manual.py`). Es idempotente: re-ejecutarla no duplica filas.

In [5]:
CREAR_POST_HIST = False   # poner True para (re)construir nessie.silver.post_hist

if CREAR_POST_HIST:
    import silver_post_hist_manual
    silver_post_hist_manual.main(spark)
else:
    print(f"{comun.TABLA_POST_HIST}: {spark.table(comun.TABLA_POST_HIST).count():,} filas")

nessie.silver.post_hist: 2,009,669 filas


In [6]:
spark.stop()